In [48]:



x_cols1 = ['IX_TOT', 'P02', 'P03', 'AGLO_rk', 'Reg_rk', 'V01', 'H05', 'H06',
       'H07', 'H08', 'H09', 'H10', 'H11', 'H12', 'H16', 'H15', 'PROP', 'H14',
       'H13', 'P07', 'P08', 'P09', 'P10', 'P05', 'CONDACT']

predecir1 = ['CAT_OCUP', 'CAT_INAC', 'CH07']

x_cols2 = x_cols1 + predecir1
predecir2 = ['INGRESO', 'INGRESO_NLB', 'INGRESO_JUB', 'INGRESO_SBS']

x_cols3 = x_cols2 + predecir2
# La seccion PP07G pregunta si el trabajo es en blanco y que beneficios tiene. Puede ayudar a la regresion para ingresos.
# predecir3 = ['PP07G1', 'PP07G2', 'PP07G3', 'PP07G4', 'PP07G_59', 'PP07H', 'PP07I', 'PP07J', 'PP07K']
predecir3 = ['PP07G1','PP07G_59', 'PP07I', 'PP07J', 'PP07K']

# Columnas de ingresos. Necesitan una regresion...
# columnas_pesos = [u'P21', u'P47T', u'PP08D1', u'TOT_P12', u'T_VI', u'V12_M', u'V2_M', u'V3_M', u'V5_M']
columnas_pesos = [u'P21', u'P47T', u'PP08D1', u'T_VI', u'V2_M']

x_cols4 = x_cols3 + predecir3
# Columnas de ingresos. Necesitan una regresion...



In [49]:
import pandas as pd
from numpy import log10

models_path = './..'
overwrite = True
startyr = 2023
endyr = 2024

yr = str(startyr)


In [50]:

import os
import joblib
import numpy as np
import pandas as pd

from numpy import log10
from pathlib import Path

from sklearn.model_selection import train_test_split

# ----------------------------
# 5) Ejecución: úsalo como reemplazo directo del RF
# ----------------------------

# Variables del usuario (mantengo tus nombres)
models_path = './..'
overwrite = True
startyr = 2023
endyr = 2024
yr = str(startyr)

# Columnas de entrada/salida ya definidas en tu entorno:
# x_cols4, columnas_pesos, etc.
# Asegurá también la lista de columnas categóricas (si aplica):
categorical_X = [
    # Ejemplos: ajustá según tu EPH
    'V01', 'H05', 'H06', 'H07', 'H08', 'H09',
       'H10', 'H11', 'H12', 'PROP', 'H14', 'H13', 'IX_TOT', 'CAT_INAC',
       'CAT_OCUP', 'P02', 'CH07', 'P07', 'P08', 'P10', 'P05', 'CONDACT',
       'PP07G1', 'PP07G_59',
       'PP07I', 'PP07J', 'PP07K', 'Reg_rk', 'INGRESO',
       'INGRESO_NLB', 'INGRESO_JUB', 'INGRESO_SBS'    # y cualquier otra que sea verdaderamente categórica
]

# Directorio de salida
out_dir = os.path.join(models_path, 'fitted_models_hgbr')



In [51]:

def prepare_Xy(df: pd.DataFrame, x_cols, y_cols, categorical_cols=None, test_size=0.1, random_state=42):
    X = df[x_cols].copy()
    y = df[y_cols].copy()

    # Categóricas nativas para HGBR (pandas dtype 'category')
    categorical_cols = categorical_cols or []
    for c in categorical_cols:
        if c in X.columns:
            X[c] = X[c].astype("category")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    return X_train, X_test, y_train, y_test


import numpy as np
import pandas as pd
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

def categorical_mask_from_df(X_df: pd.DataFrame) -> np.ndarray:
    return np.array([pd.api.types.is_categorical_dtype(X_df[c]) for c in X_df.columns], dtype=bool)

def make_hgbr_with_mask(cat_mask, **overrides):
    params = dict(
        max_depth=10,
        learning_rate=0.05,
        max_iter=500,
        l2_regularization=1.0,
        early_stopping=True,
        categorical_features=cat_mask,
        random_state=42,
    )
    params.update(overrides)
    return HistGradientBoostingRegressor(**params)

def make_multioutput_hgbr_with_mask(cat_mask):
    return MultiOutputRegressor(make_hgbr_with_mask(cat_mask))


def make_multioutput_log1p_with_mask(cat_mask):
    base = make_hgbr_with_mask(cat_mask)  # your mask-based HGBR factory
    wrapped = TransformedTargetRegressor(
        regressor=base,
        transformer=FunctionTransformer(np.log1p, np.expm1, validate=False),
        check_inverse=False,
    )
    return MultiOutputRegressor(wrapped)

def assert_all_finite(name, arr):
    if isinstance(arr, pd.DataFrame):
        mask = ~np.isfinite(arr.to_numpy())
    else:
        mask = ~np.isfinite(arr)
    if mask.any():
        idx = np.argwhere(mask)[:10]
        raise ValueError(f"{name} contains non-finite values at indices (first 10): {idx.tolist()}")


In [52]:

print(yr)
train_data = pd.read_csv('./../../data/training/EPHARG_train_'+yr[2:]+'.csv')


2023


In [53]:
# A) Plain HGBR baseline – keep your pre-log10
train_prelog = train_data.copy()
train_prelog[columnas_pesos] = np.log10(train_prelog[columnas_pesos].clip(lower=0) + 1)

train_data = train_prelog 

# X_train, X_test, y_train, y_test = prepare_Xy(
#     train_prelog, x_cols=x_cols4, y_cols=columnas_pesos,
#     categorical_cols=categorical_X, test_size=0.1, random_state=42
# )
# cat_mask = categorical_mask_from_df(X_train)
# # You will train a plain HGBR on already log10-transformed targets.

# # B) TTR path – use RAW pesos (NO upstream log)
# train_raw = train_data.copy()
# train_raw[columnas_pesos] = train_raw[columnas_pesos].clip(lower=0)

# X_train_raw, X_test_raw, y_train_raw, y_test_raw = prepare_Xy(
#     train_raw, x_cols=x_cols4, y_cols=columnas_pesos,
#     categorical_cols=categorical_X, test_size=0.1, random_state=42
# )
# cat_mask_raw = categorical_mask_from_df(X_train_raw)


In [54]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.multioutput import MultiOutputRegressor

def make_multioutput_log1p_with_mask(cat_mask):
    base = make_hgbr_with_mask(cat_mask)  # your mask-based HGBR factory
    wrapped = TransformedTargetRegressor(
        regressor=base,
        transformer=FunctionTransformer(np.log1p, np.expm1, validate=False),
        check_inverse=False,
    )
    return MultiOutputRegressor(wrapped)


In [55]:

import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def ensure_dir(p: str): Path(p).mkdir(parents=True, exist_ok=True)


def print_metrics(tag, rows):
    print(f"\n=== {tag} ===")
    for r in rows:
        print(f"{r['target']:10s} | R²={r['r2']:6.3f} | RMSE={r['rmse']:7.3f} | MAE={r['mae']:7.3f}")



In [56]:
def make_hgbr_base_for_chain(**overrides):
    # no categorical_features here; chain feeds numpy arrays
    params = dict(
        max_depth=10,
        learning_rate=0.05,
        max_iter=200,
        l2_regularization=1.0,
        early_stopping=True,
        random_state=42,
    )
    params.update(overrides)
    return HistGradientBoostingRegressor(**params)


def fit_eval_save(model, X_train, y_train, X_test, y_test, out_dir, tag):
    fitted = model.fit(X_train, y_train)
    y_pred = fitted.predict(X_test)

    # guard: identify non-finite predictions per target and skip them
    y_pred = np.asarray(y_pred)
    y_true = y_test.values
    names = y_test.columns

    rows = []
    for j, name in enumerate(names):
        col_pred = y_pred[:, j]
        col_true = y_true[:, j]
        if not np.isfinite(col_pred).all():
            bad_idx = np.where(~np.isfinite(col_pred))[0][:5].tolist()
            print(f"[WARN] {tag}: {name} produced non-finite predictions. "
                  f"Examples at rows {bad_idx}. Skipping metrics for this target.")
            continue
        rows.append({
            "target": name,
            "r2": float(r2_score(col_true, col_pred)),
            "rmse": float(mean_squared_error(col_true, col_pred, squared=False)),
            "mae": float(mean_absolute_error(col_true, col_pred)),
        })

    print_metrics(tag, rows)

    ensure_dir(out_dir)
    model_path = os.path.join(out_dir, f"{tag}.joblib")
    joblib.dump(fitted, model_path, compress=3)
    print(f"Modelo guardado en: {model_path}")
    return fitted, rows



# 2) Chain (uses one-hot path; no native cats here)
def make_per_target_chain(order=None):
    base = make_hgbr_base_for_chain()
    from sklearn.multioutput import RegressorChain
    return RegressorChain(base_estimator=base, order=order)


from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- new: encoder util that returns train/test numeric matrices ---
def prepare_Xy_onehot(df, x_cols, y_cols, categorical_cols=None, test_size=0.1, random_state=42):
    X = df[x_cols].copy()
    y = df[y_cols].copy()

    categorical_cols = [c for c in (categorical_cols or []) if c in X.columns]
    numeric_cols = [c for c in X.columns if c not in categorical_cols]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # column transformer that one-hots cats and passes numeric through
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    pre = ColumnTransformer(
        transformers=[
            ("cat", ohe, categorical_cols),
            ("num", "passthrough", numeric_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    # fit only on train; transform both
    X_train_enc = pre.fit_transform(X_train)
    X_test_enc  = pre.transform(X_test)

    return X_train_enc, X_test_enc, y_train, y_test, pre, (categorical_cols, numeric_cols)


In [57]:
Xtr_enc, Xte_enc, ytr_enc, yte_enc, pre_enc, _ = prepare_Xy_onehot(
    train_data.sample(frac = .2), x_cols=x_cols4, y_cols=columnas_pesos,
    categorical_cols=categorical_X, test_size=0.1, random_state=42
)

order = list(range(len(columnas_pesos)))
chain_hgbr = make_per_target_chain(order=order)   # base HGBR without categorical_features
chain_model, chain_metrics = fit_eval_save(
    chain_hgbr, Xtr_enc, ytr_enc, Xte_enc, yte_enc, out_dir, tag="HGBR_chain"
)
joblib.dump(pre_enc, os.path.join(out_dir, "HGBR_chain_preprocessor.joblib"), compress=3)



=== HGBR_chain ===
P21        | R²= 0.961 | RMSE=  0.351 | MAE=  0.121
P47T       | R²= 0.988 | RMSE=  0.203 | MAE=  0.149
PP08D1     | R²= 0.919 | RMSE=  0.473 | MAE=  0.154
T_VI       | R²= 0.990 | RMSE=  0.152 | MAE=  0.101
V2_M       | R²= 0.997 | RMSE=  0.077 | MAE=  0.023
Modelo guardado en: ./../fitted_models_hgbr/HGBR_chain.joblib


['./../fitted_models_hgbr/HGBR_chain_preprocessor.joblib']

In [62]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

# --- predictions on held-out test split -------------------
# (use the model actually trained — here: chain_model)
y_pred = chain_model.predict(Xte_enc)
y_true = yte_enc.values
target_names = yte_enc.columns

# --- global metrics ---------------------------------------
print("\n=== Metrics on held-out test ===")
for j, name in enumerate(target_names):
    r2 = r2_score(y_true[:, j], y_pred[:, j])
    rmse = mean_squared_error(y_true[:, j], y_pred[:, j], squared=False)
    print(f"{name:10s} | R²={r2:6.3f} | RMSE={rmse:8.3f}")

# --- 1. scatter: y_true vs y_pred per variable ------------
ncols = 3
nrows = int(np.ceil(len(target_names) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))

for j, name in enumerate(target_names):
    ax = axes.flat[j]
    ax.scatter(y_true[:, j], y_pred[:, j], alpha=0.3, s=10)
    lims = [
        np.nanmin([y_true[:, j], y_pred[:, j]]),
        np.nanmax([y_true[:, j], y_pred[:, j]])
    ]
    ax.plot(lims, lims, 'r--', lw=1)
    ax.set_title(name)
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")

# remove empty subplots
for k in range(len(target_names), nrows * ncols):
    fig.delaxes(axes.flat[k])

plt.tight_layout()
plt.show()


# --- 2. residuals vs fitted -------------------------------
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 5*nrows))
for j, name in enumerate(target_names):
    ax = axes.flat[j]
    resid = y_true[:, j] - y_pred[:, j]
    ax.scatter(y_pred[:, j], resid, alpha=0.3, s=10)
    ax.axhline(0, color='r', lw=1)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Residual")

for k in range(j+1, nrows*ncols):
    fig.delaxes(axes.flat[k])

plt.tight_layout()
plt.show()
